## ModelEU27
Powered by [Eleonora Priori](https://eleonorapriori.github.io/website/) and [Pietro Terna](https://terna.to.it/) 


In [1]:
def making924table(seed, rankNum):
    tab = [[0] * rankNum for _ in range(924)]
    random.seed(seed)
    for i in range(924):
        tab[i][random.randint(0,rankNum-1)] = 1
        
    return tab   

In [2]:
from mpi4py import MPI
from repast4py import context as ctx
import repast4py 
from repast4py import parameters
from repast4py import schedule
from repast4py import core
from typing import Tuple, List, Dict
import numpy as np
import pandas as pd
import pickle
import random
import csv
import os
import sys
import time
import sectors
from itertools import accumulate

comm = MPI.COMM_WORLD
rank    = comm.Get_rank()
rankNum = comm.Get_size() 

if rankNum > 0: print('rank', rank, 'starting',flush=True)

# create the context to hold the agents and manage cross process
# synchronization
context = ctx.SharedContext(comm)

# Initialize the default schedule runner, HERE to create the t() function,
# returning the tick value
runner = schedule.init_schedule_runner(comm)

#Initializes the repast4py.parameters.params dictionary with the model input parameters.
params = parameters.init_params("modelEU27.yaml", "")

#making the table splitting the creation of firms in ranks
seed924 = params['seed924'] #special independent seed for the 915 table
tab924 = making924table(seed924,rankNum)

#generate random seed
repast4py.random.init(rng_seed=params['myRandom.seed'][rank]) #each rank has a seed
rng = repast4py.random.default_rng # repast seed
rng_np = np.random.default_rng(params['myRandom.seed'][rank]) # numpy seed
random.seed = params['myRandom.seed'][rank]

#timer T()
startTime=-1
def T():
    global startTime
    if startTime < 0:
        startTime=time.time()
    return time.time() - startTime
T() #launches the timer

#cpuTimer Tc()
startCpuTime=-1
def Tc():
    global startCpuTime
    if startCpuTime < 0:
        startCpuTime=time.process_time()
    return time.process_time() - startCpuTime
Tc() #launches the cpu timer

#read control key from firm-features-generation
with open("control4text_ff.txt", "r") as f:
    control = int(f.read())

rank 0 starting


In [3]:
class Firm(core.Agent):

    TYPE = 0
    
    def __init__(self, local_id: int, rank: int, sector: int, labor: int, capital: float, capitalR: float, wage: float,\
                 country: int, recipe: float, countryRecipe: float, laborProductivity: float, avgAssetsUsefulLife: float,\
                 avgPlannedMarkup: float, productionType: str):
                 # orderObservationFrequency: int, 
                 
        super().__init__(id=local_id, type=Firm.TYPE, rank=rank) #uid
    
        self.sector=sector 
        self.labor=labor
        self.capital=capital
        self.capitalR=capitalR
        self.wage=wage
        self.country = country
        #self.investment = investment # serviva per interagire col planner ?
        self.recipe = recipe
        self.countryRecipe = countryRecipe
        self.laborProductivity=laborProductivity
        self.assetsUsefulLife=avgAssetsUsefulLife
        self.plannedMarkup=avgPlannedMarkup
        self.productionType = productionType
        self.outputWarehouse = 0
        self.intermediateWarehouse = [0] * 64
        self.intermediateWarehouseRefill= [0] * 64
        self.targetAddedValue = 0
        self.actualAddedValue = 0 

        # here we introduce an estimated value of added value in t-1 to generate the intermediate warehouse quantities available at t=0
        self.estimatedAddedValueTminus1 = float(self.labor * self.wage + self.capital * 1000 * self.capitalR) / 12
        self.estimatedAddedValueTminus1 *= self.plannedMarkup
        
        # self.orderObservationFrequency=orderObservationFrequency

In [4]:
class EUstat(core.Agent):

    TYPE = 1
    
    def __init__(self, local_id: int, rank: int): 
        super().__init__(id=local_id, type=EUstat.TYPE, rank=rank) #uid

        self.rankResults = [] # it will be filled in fillingRankResults
        
    def save(self) -> Tuple:
        return (self.uid, 
                self.rankResults
               )
        
    def update(self, dynState) -> None:
        #for i in range(len(self.rankResults)): # self.rankResults is a list to be filled on the way by appending the required items
           self.rankResults = dynState


In [5]:
# Local cache: avoids recreating agents already present on this rank
agent_cache = {}  # Dict[uid, Agent]

def restore_agent(agent_data: tuple): 
    """
    Reconstructs or updates an agent from serialized data.
    It is the complement of `save()` and is passed to context.synchronize(...).

    agent_data: a tuple in the form (uid, state_tuple)
    where uid is typically (owner_rank, TYPE, local_id)
    and state_tuple is the tuple returned by save().    
    """
    uid, state = agent_data
    agent_type = uid[1]  # conventionally: uid = (rank, TYPE, local_id)

    # EUstat
    if agent_type == EUstat.TYPE:
        if uid in agent_cache:
            ag = agent_cache[uid]
            # Consistency with the template: we ALWAYS use update(dynState)
            
        else:
            # Ricreazione da zero rispettando l'ordine della tupla di save()
            # a = state 
            ag = EUstat(uid[0],uid[2])
            agent_cache[uid] = ag

        ag.update(state)
            
        return ag

    # If unknown individuals arrive, explicitly fail
    raise ValueError(f"restore_agent: unknown type in uid {uid}")

In [6]:
def statistics(y_true, y_est):

    y_true = np.array(y_true)
    y_est  = np.array(y_est)

    n = y_true.shape[0]
    
    #MAE, Mean Absolute Error
    # $ MAE = \frac{1}{n}\sum_{i=1}^n | yi - \hat{y}_i | $
    mae = abs(y_true - y_est).mean()

    # RMSE, Root Mean Squared Error
    # $ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n ( y_i - \hat{y}_i )^2} $
    rmse = (((y_true - y_est)**2).sum()/n)**0.5

    # MAPE, Mean Absolute Percentage Error
    # $ MAPE = \frac{100}{n}\sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right| $
    safeDiv = np.where(y_true == 0, 0.000001, 0)
    mape = (abs((y_true - y_est) / (y_true+safeDiv)).sum()) * 100 / n

    # R², coefficient of determination
    # $ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $
    if ((y_true - y_true.mean())**2).sum() == 0: r2 = 0
    else: r2 = 1 - (((y_true - y_est)**2).sum()/((y_true - y_true.mean())**2).sum())**0.5

    return (mae,rmse,mape,r2,n)  #,rmse, mape,r2,count)

  
  
$ MAE = \frac{1}{n}\sum_{i=1}^n | yi - \hat{y}_i | $

$ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n ( y_i - \hat{y}_i )^2} $

$ MAPE = \frac{100}{n}\sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right| $

$ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $


In [7]:
from IPython.display import display, HTML
def stop_execution(message):
    display(HTML(f"<pre>{message}</pre>"))
    raise SystemExit(0)

In [8]:
def generateOutput(countryLabel, modelOutput, AFF_FirmNumber, countAFF_Firms, propFactor, AFF_WorkerNumber, countAFF_WorkerNumber,
                   Eu_FirmNumber, countActualExPostFirmNumber, Eu_WorkerNumber, countWorkers, Eu_EmployeeCompensations,
                   countEmployeeCompensations, Eu_GDP_withVAT2022, Eu_AddedValue2022, countCapitalR, countActualAddedValue, markupTentative,
                   Eu_Intermediate2022, intermediateTotal, substitutionRate, substitutionRateL, firmSubstitutions,
                   allfirmSubstitutionsByBuyerSector, simulationFirmsExAnteNumber, allfirmSubstitutionsByVendorSector):
    

    # employeeCompensations_CapitalR_markupAugmented = \
    #   int(markupTentative*(countEmployeeCompensations/1_000_000 + countCapitalR/1000))
    capitalR_fullMarkupAugmented = \
       int(countCapitalR/1000 + (markupTentative-1)*(countEmployeeCompensations/1_000_000 + countCapitalR/1000))
    table_data = [
        [' ',          'EU 27', 'simulation', 'simulation to'],
        [' ',          ' ',     ' ',          'EU scale'],
        ['AFF Firm Number', AFF_FirmNumber, countAFF_Firms, int(countAFF_Firms * propFactor)], 
        ['AFF Worker Number', AFF_WorkerNumber, countAFF_WorkerNumber, int(countAFF_WorkerNumber * propFactor)], 
        ['Firm Number', Eu_FirmNumber, countActualExPostFirmNumber, int(countActualExPostFirmNumber * propFactor)],
        ['Worker Number', Eu_WorkerNumber, countWorkers, int(countWorkers * propFactor)],
        ['Empl. Compens.', int(Eu_EmployeeCompensations), int(countEmployeeCompensations/1_000_000), 
                           int(countEmployeeCompensations * propFactor / 1_000_000)],                                                                                                                              
        ['Cap. R & Op. Surplus', int(Eu_AddedValue2022 - Eu_EmployeeCompensations), capitalR_fullMarkupAugmented, 
                                 int(capitalR_fullMarkupAugmented*propFactor)],
        ['GDP curr. p. (VAT)', Eu_GDP_withVAT2022, 'N.B. contains VAT',' '],
        ['Add. Val. basic p.', Eu_AddedValue2022, int(countActualAddedValue/1_000_000), 
                                # countActualAddedValue was employeeCompensations_CapitalR_markupAugmented in the previous version
                               int((countActualAddedValue/1_000_000)*propFactor)],
        ['Int. Prod. basic p.', int(Eu_Intermediate2022), int(intermediateTotal/1000_000),
         int(intermediateTotal * propFactor / 1000000)]
    ]
    
    print("\n", countryLabel, "\n", file=modelOutput)
    
    print(f"Markup tentative {100*(markupTentative-1)}%", file=modelOutput)
    for row in table_data:
        print("{: >20} {: >20} {: >20} {: >20}".format(*row), file=modelOutput)

    print("\n AFF => Agriculture, Forestry and Fishing\n\n", file=modelOutput)


In [9]:
def generateGDPOutput(countryLabel, modelOutput, propFactor, countActualAddedValue):
                        #countEmployeeCompensations, countCapitalR, markupTentative):
    
    print(countryLabel, int((countActualAddedValue/1_000_000) * propFactor),
          #int(markupTentative*(countEmployeeCompensations * propFactor/1_000_000 + countCapitalR * propFactor/1000)), \
          file=modelOutput)

In [10]:
def scalarSummingResultsOverRanks(n):
    tot = 0
    for i in range(1, rankNum):
        rankEUstat_ghost = context.ghost_agent((0,1,i))
        tot += rankEUstat_ghost.rankResults[n]
    return tot

def summingResultsOverRanks(n, L):
    tot = [0] * L
    for i in range(1, rankNum):
        rankEUstat_ghost = context.ghost_agent((0,1,i))
        for j in range(L):
            tot[j] += rankEUstat_ghost.rankResults[n][j]
    return tot

def summingNumpyMatrixResultsOverRanks(n):
    tot = np.zeros([65,28])
    for i in range(1,rankNum):
        rankEUstat_ghost = context.ghost_agent((0,1,i))
        for j in range(28):
            for k in range(65):
                tot[k,j] += rankEUstat_ghost.rankResults[n][k,j] 
    return tot

def summingMatrixResultsOverRanks(n):
    tot = [[0] * 28 for _ in range(64)]
    for i in range(1,rankNum):
        rankEUstat_ghost = context.ghost_agent((0,1,i))
        for j in range(28):
            for k in range(64):
                tot[k][j] += rankEUstat_ghost.rankResults[n][k][j] 
    return tot


def scalingWorkerDistribution(worker_distribution):
    values = [0] * 1849
    # class 1-9
    tot = 0
    for i in range(9): 
        tot += worker_distribution[i]
    for i in range(9):
        values[i] = worker_distribution[i] / tot
    # class 10-19
    tot = 0 
    for i in range(9, 19):
        tot += worker_distribution[i]
    for i in range(9, 19):
        values[i] = worker_distribution[i] / tot
    # class 20-49
    tot = 0 
    for i in range(19, 49):
        tot += worker_distribution[i]
    for i in range(19, 49):
        values[i] = worker_distribution[i] / tot
    # class 50-249
    tot = 0 
    for i in range(49, 249):
        tot += worker_distribution[i]
    for i in range(49, 249):
        values[i] = worker_distribution[i] / tot
    # class 250 or more (1849)
    tot = 0 
    for i in range(249, 1849):
        tot += worker_distribution[i]
    for i in range(249, 1849):
        values[i] = worker_distribution[i] / tot     
    return values


class Model:
    
    def __init__(self):
        

        #ghosts' list or lists
        self.EUstatGhostList=[] #used only if rank == 0 but cannot be in an if to be seen below for sure
        self.count_months = -1
        self.propFactor = 0
        
        # SCHEDULE
        runner.schedule_event(          0.0,     self.initGhosts) 
        runner.schedule_repeating_event(0.1,  1, self.supply)
        runner.schedule_repeating_event(0.2,  1, self.demand)
        runner.schedule_repeating_event(0.3,  1, self.aggregatingResultsByCountry)
        runner.schedule_repeating_event(0.35, 1,self.selectSuppliers)
        
        runner.schedule_repeating_event(0.4,  1, self.fillingRankResults)
        runner.schedule_repeating_event(0.5,  1, self.synchronizeRanks)
        runner.schedule_repeating_event(0.6,  1, self.aggregatingRanks)
        runner.schedule_repeating_event(0.7,  1, self.showResults)

        #runner.schedule_stop(0.8) # N x 12 identifies the number of years to be run as one time-step is one month
        runner.schedule_stop(params['howManyCycles']) # N x 12 identifies the number of years to be run 
                                                      # as one time-step is one month
        runner.schedule_end_event(self.finish)

        self.eu_countries = ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 
                             'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 
                             'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 
                             'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 
                             'Slovenia', 'Spain', 'Sweden', 'totEU27']
        country_numid_list = list(range(27))

        self.selected_countries = params['selected_countries']
    
        eu_productivities = pd.read_pickle("./eu_productivities.xp") # generated in computing_productivity.ipynb
        self.adjustingCoefficient = eu_productivities['no agri tot prod Italy-adjusted'] # adjustment index, based on Istat data on capital 
        self.agri_adjustingCoefficient = eu_productivities['agri tot prod']  
        nama_av = pd.read_pickle("./nama_av.xp") # generated in computing_productivity.ipynb
        self.shareConsumptionDemandTable = pd.read_pickle("./shareConsumptionDemand.xp")  # generated in initialConsumptionDemand.ipynb

        self.lc_countries = pd.read_pickle("./lc_countries.xp") # generated in lc_countries.ipynb
        # lc_countries is a column of the labor cost index by country
        
        self.gdp27_comparison = pd.read_pickle("./gdp27_comparison.xp") # generated in GNP27basicPrices.ipynb
        # at the moment here only for comparison reasons (should not be relevant to new steps of the model)
        
        self.consumptionShares = pd.read_pickle("./shareConsumptionDemand.xp") # generated in initialConsumptionDemand.ipynb

        #print(self.consumptionShares.iloc[2,-2],self.consumptionShares.iloc[2,-1])
        #self.consumptionShares = self.consumptionShares.set_index("country")
        #print(self.consumptionShares.at["Bulgaria", "private_cons / gdp"], self.consumptionShares.at["Bulgaria", "gov_cons /gdp"])

        
        ####################################################################################################
        ###################################### CREATE EUstat AGENT ##########################################
        ####################################################################################################

        if rank != 0:
            rankEUstat = EUstat (0, rank)
        
            context.add(rankEUstat)
        
        
        ####################################################################################################
        ###################################### CREATE FIRM AGENTS ##########################################
        ####################################################################################################

        with open('naio_io_N.xp', 'rb') as f:
            intermediateInputs_float64 = pickle.load(f) #numpy array, only numeric, see last row of firm-features-generation.ipynb
                                                #eliminating nan in col 43 (used), not in 64 (never used)
        self.intermediateInputs = intermediateInputs_float64.astype(float).tolist()
                                                # from now onwards we have to call its values by using * [i][j] * as indexes
        
        #firm investment shares
        nama = params['tableOfInvestmentShares']
        print('rank', rank, 'tableOfInvestmentShare', nama)

        if nama==0: self.invShares = pd.read_pickle("./invTableNoNama.xp") #ignoring nama infos
        if nama==1: self.invShares = pd.read_pickle("./invTableNama.xp")   #using nama infos
        if nama==2: self.invShares = pd.read_pickle("./invTableNamaIPF.xp")   #using nama infos

        self.invSharesNp = self.invShares.to_numpy()
        #print(self.invSharesNp)

        #65 rows x 66 columns
        #print (self.invShares)
        #print (self.invShares.iloc[0,0]) #Crop
        #print (self.invShares.iloc[1,0]) #Forest
        #print (self.invShares.iloc[0,1]) #0.002242 values for invTableNoNama (here and following)
        #print (self.invShares.iloc[1,1]) #0.002242
        #print (self.invShares.iloc[0,2]) #0.000131
        #print (self.invShares.iloc[1,2]) #0.000131
        #print (self.invShares.iloc[43,0]) #Imputed 
        #print (self.invShares.iloc[44,0]) #Real
            
        for i in range(64): #64, as rows 0:63
            self.intermediateInputs[i][43]=0
            #self.intermediateInputs[i][64]=0 #col 64 never user, as sector 65 was dropped

        fileName = "ff_with_class_limits.csv" #input("file name? ")

        # how many firms
        #self.simulationFirmsExAnteNumber = int(input("how many firms? "))
        self.simulationFirmsExAnteNumber = params['howManyFirms']
        print('rank', rank, 'how many firms', self.simulationFirmsExAnteNumber)

        with open("workers_distribution.xp", "rb") as f:
            worker_distribution = pickle.load(f)

        scaled_worker_distribution = scalingWorkerDistribution(worker_distribution) 
        L  = [(0,9), (9,19), (19,49), (49,249), (249,1849)] #probabilities
        #scaled_worker_distribution[L[cl][0]:L[cl][1]] to call specific values within a class
        wN = [i for i in range(1,1850)]
        #wN[L[cl][0]:L[cl][1]] to call specific values within a class
        
        # smart capital
        """
        self.smart_capital = 2
        while self.smart_capital != 0 and self.smart_capital != 1:
            self.smart_capital = int(input("use smart capital? (Please, use only 1/0 to express True/False values) "))
            if self.smart_capital != 0 and self.smart_capital != 1: print("Please, use only 1/0 to express True/False values")
        """
        self.smart_capital = params['useSmartCapital']
        print('rank', rank, 'use smart capital', self.smart_capital)

        # from addedvalue_by_country.ipynb - in folder structuralData
        with open('addedvalue_countryshares.xp', 'rb') as f:
            addedvalue_countryshares = pickle.load(f)
            countryshare_list = addedvalue_countryshares['Share in 2020'].tolist()
            # addedvalue_countryshares['Country', 'Share in 2020'] where 'Country' is an index, not a column
            # to call the share use addedvalue_countryshares.iloc[0,x]
            # can also call by country doing addedvalue_countryshares.at['Austria', 'Share in 2020']

        # from sbs_countries.ipynb
        eu_firms_by_employed_number = pd.read_pickle("./eu_firms_by_employed_number.xp") # this is a list of dfs (0-9, 10-19,20-49,50-249,>250)
        sector_special_cases = ['1', '2', '3', '44'] # agri-silvi-fish and imputed rents
        agri_eu27 = pd.read_pickle("./agri_eu27base.xp") # agri-silvi-fish shares (countries are cols to keep 
                                                        # consistency with eu_firms_by_employed_number)
                                                       # the agri_eu27base file comes from ./special_sectors_eu27.ipynb (cell 7)
        agri_eu27.index = ["Agricultural share", "Forestry share", "Fishing share"]

        prob_country_agri = agri_eu27.loc["Agricultural share"].tolist()
        prob_country_silvi = agri_eu27.loc["Forestry share"].tolist()
        prob_country_fishing = agri_eu27.loc["Fishing share"].tolist()

        self.eu27_owners = pd.read_pickle("./eu27_ownership.xp") # 44 sector, imputed rents from houses ownership by EU country
                                                                 # generated from special_sectors_EU27.ipynb

        #importing csv file containing info about firms 
        with open(fileName, newline='') as csvfile:
            firmReader= csv.reader(csvfile, delimiter=',')#, quoting=csv.QUOTE_NONNUMERIC)
    
            self.rowNumber=-1 #to skip the row of the headers
            k=0 
            createImputedRentsFirms = True
            if rank != 0: createImputedRentsFirms = False # to avoid firms creation from sect44 in ranks != 0
            
            for row in firmReader: #for each record in .csv
                #print(row)
                labor = 0 
                if self.rowNumber >= 0 and tab924[self.rowNumber][rank] == 1: # skip if self.rowNumber == -1 or tab924[][] == 0
                    if row[4]=='': row[4]=0 # it pertains with last rows (naio sector 65)                       
                    sector = row[0]

                    #*****************************************************************************************
                    initialFirmCreationNumber= round(float(row[4]) * self.simulationFirmsExAnteNumber)
                    firmCreationNumber = initialFirmCreationNumber 
                    
                    if sector =='44' and createImputedRentsFirms:
                        firmCreationNumber= 27 # number of EU countries
                        createImputedRentsFirms = False

                    countImputedRentsFirms = 0

                    #*****************************************************************************************
                    # class attribution
                    if row[3] == "From 0 to 9 persons employed":    cl = 0
                    if row[3] == "From 10 to 19 persons employed":  cl = 1
                    if row[3] == "From 20 to 49 persons employed":  cl = 2
                    if row[3] == "From 50 to 249 persons employed": cl = 3
                    if row[3] == "250 persons employed or more":    cl = 4

                            
                    for i in range(firmCreationNumber):
                        if row[0] == '1' or row[0] == '2' or row[0] == '3': #agri, silvi and fishing
                            randomizer = rng.uniform()
                            if randomizer <= 0.2: labor = 0
                            elif randomizer <= 0.9 : labor = 1
                            else: labor = 2
                        else:
                            # use L and scaled_worker_distribution
                            labor = rng_np.choice(wN[L[cl][0]:L[cl][1]], p=scaled_worker_distribution[L[cl][0]:L[cl][1]])
                            
                            if sector =='44': labor = 0 # no employees in a virtual sector
                        
                        
                        # assignation of EU country label/number 
                        country_randomizer = rng.uniform()
                        # assigning rule: we attribute the first country with a cumulative share higher than the random number we draw
                        # non-special cases sectors come from the sbs_countries file, while agriculture sector is treaten in the agri_eu27 file
                        country_counter = 0
                        #if not row[0] in sector_special_cases:
                        if not row[0] in sector_special_cases and int(row[0]) != 61 and int(row[0]) != 64:
                            #if eu_firms_by_employed_number[cl].iloc[int(row[0]), -1] != 0: # to avoid sectors with all zeros 
                                #while eu_firms_by_employed_number[cl].iloc[int(row[0]), country_counter] <= country_randomizer: country_counter += 1
                            prob_country_sec_cl = eu_firms_by_employed_number[cl].loc[int(row[0])].squeeze().tolist()

                            country_counter = rng_np.choice(country_numid_list, p=prob_country_sec_cl)  
                         
                        
                        elif row[0] == '44' and rank == 0: # imputed rents
                            country_counter = countImputedRentsFirms
                            countImputedRentsFirms += 1

                        elif int(row[0]) == 61 or int(row[0]) == 64: # 61: membership organizations, 64: services by households
                            country_counter = rng_np.choice(country_numid_list, p=countryshare_list) 
                            
                                
                        else: # agri forestry and fishing
                            if int(row[0]) == 1: country_counter = rng_np.choice(country_numid_list, p=prob_country_agri)
                            if int(row[0]) == 2: country_counter = rng_np.choice(country_numid_list, p=prob_country_silvi)
                            if int(row[0]) == 3: country_counter = rng_np.choice(country_numid_list, p=prob_country_fishing)

                   
                        capital= float(row[11]) + rng.random()*(float(row[12]) - float(row[11]))
                        if self.smart_capital:
                            capital = labor * float(row[10]) # row[10] = recipe 
                            
                        capitalR = float(row[13])
                        wage = float(row[14])

                        # adjusting wage and capital accounting for differences in national productivity
                        # the adjustingCoefficient is a vector containing indeces of national productivity (for agri and no-agri sectors)
                                      # for all the countries (ordered in the standard order in which we report the list of EU countries)
                        
                        if row[0] == '1' or row[0] == '2' or row[0] == '3': #agri, silvi and fishing
                            wage *= self.agri_adjustingCoefficient[country_counter]
                            capital *= self.agri_adjustingCoefficient[country_counter]
                        else:
                            #wage *= self.adjustingCoefficient[country_counter]
                            wage *= self.lc_countries[country_counter]
                            capital *= self.adjustingCoefficient[country_counter]
                        
                        
                        # !!!
                        # for the intermediate inputs each firm must grabble its values directly from the self.intermediateInputs table
                        # by using self.intermediateInputs address which is internal to the model

                        recipe = float(row[10])
                        countryRecipe = recipe * self.adjustingCoefficient[country_counter]
                        
                        if row[0] == '1' or row[0] == '2' or row[0] == '3': 
                            laborProductivity = nama_av['agri_tot_prod'][country_counter]
                        else: 
                            laborProductivity = nama_av['noagri_tot_prod'][country_counter]
                         
                        avgAssetsUsefulLife = 12*12 # 12 years, now supposed to be equal for all the sectors (except real estate sector)
                        avgPlannedMarkup = params['plannedMarkup']
                        productionType = row[2]
                        #orderObservationFrequency=rng.integers(row[12], row[13]+1)
                        
                        aFirm =Firm(k, rank, int(sector), labor, capital, capitalR, wage, country_counter,\
                                    recipe, countryRecipe, laborProductivity, avgAssetsUsefulLife, avgPlannedMarkup, productionType) 
                                    #orderObservationFrequency
                        # here we pass "self" as the address of the model to provide all firms with the self.intermediateInputs table
                        
                        context.add(aFirm)
                        k += 1 # first element of the UID of the agents
                        #print(aFirm.uid, aFirm.sector, aFirm.country)

                        #memo: in firm _init_ we have estimatedAddedValueTminus1
                        # Creating t=0 intermediate warehouse using estimated added value at t-1
                        for j in range(64):
                            aFirm.intermediateWarehouse[j] = self.intermediateInputs[j][aFirm.sector-1] * aFirm.estimatedAddedValueTminus1
             
                        
                self.rowNumber += 1
                self.simulationFirmsExPostNumber=k #one more, here is a count, not an id
                
            print(self.simulationFirmsExPostNumber, "firms created in rank ", rank)

            


        
        self.countActualExPostFirmNumber = len(list(context.agents(agent_type=0)))
        
        self.AFF_FirmNumber = 10000000
        self.Eu_FirmNumber = 32251874 + self.AFF_FirmNumber
        self.AFF_WorkerNumber = 9000000
        self.Eu_WorkerNumber = 162467679 + 9000000
        self.Eu_EmployeeCompensations = 7447036.79
        self.Eu_GDP_withVAT2022 = 16144780 #https://ec.europa.eu/eurostat/databrowser/view/tec00001/default/table?lang=en
        self.Eu_AddedValue2022 = 14303899 #naio table 2022 (milions) total Added value, gross
        self.Eu_Intermediate2022 = 16939701.18 	 
        if self.propFactor == 0: self.propFactor = self.Eu_FirmNumber / self.countActualExPostFirmNumber
        print("propFactor", self.propFactor,"in rank",rank)


        # creating all firms database by sector, one global and one for each productionType
        self.firmListBySector =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeC   =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeI   =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeInt =  [[] for _ in range(64)]  

        for aFirm in context.agents(agent_type=0):
            self.firmListBySector[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="C":   self.firmListBySectorWithProductionTypeC[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="I":   self.firmListBySectorWithProductionTypeI[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="Int": self.firmListBySectorWithProductionTypeInt[aFirm.sector-1].append(aFirm)


        #TEST
        numero_liste = len(self.firmListBySectorWithProductionTypeC)
        print(f"Ci sono {numero_liste} liste.")

        # Contare gli elementi di ciascuna lista
        tot=0
        for i, sottolista in enumerate(self.firmListBySectorWithProductionTypeC):
                print(f"La lista {i+1} contiene {len(sottolista)} elementi.")
                tot+=len(sottolista)
        print("totale sottoliste C", tot)

        #TEST
        numero_liste = len(self.firmListBySectorWithProductionTypeI)
        print(f"Ci sono {numero_liste} liste.")

        # Contare gli elementi di ciascuna lista
        tot=0
        for i, sottolista in enumerate(self.firmListBySectorWithProductionTypeI):
                print(f"La lista {i+1} contiene {len(sottolista)} elementi.")
                tot+=len(sottolista)
        print("totale sottoliste I", tot)

        #TEST
        numero_liste = len(self.firmListBySectorWithProductionTypeInt)
        print(f"Ci sono {numero_liste} liste.")

        # Contare gli elementi di ciascuna lista
        tot=0
        for i, sottolista in enumerate(self.firmListBySectorWithProductionTypeInt):
                print(f"La lista {i+1} contiene {len(sottolista)} elementi.")
                tot+=len(sottolista)
        print("totale sottoliste Int", tot)

        1/0
    
    #create ghosts
    def initGhosts(self):

        #EUstat ghost, to be pulled from non 0 ranks and to be sent to the 0 rank
        # print("rank",rank,"initGhosts1",flush=True)

        ghostsToRequest = [] # list of tuples containing for each ghost the uid and its rank;

        if rank == 0:
            for rankId in range(1,rankNum):
                ghostsToRequest.append( ((0,EUstat.TYPE,rankId),rankId) )

        # print("rank",rank,"initGhosts2",flush=True)
        
        #create ghosts and pull them
        context.request_agents(ghostsToRequest,restore_agent)

        # print("rank",rank,"initGhosts3",flush=True)
        
        #the list of central planner ghosts in rank 0 (the for cycle does not work if rankNum==1)
        if rank == 0:
            for i in range(1,rankNum):
                self.EUstatGhostList.append(agent_cache[(0,EUstat.TYPE,i)])

        print("rank",rank,"self.EUstatGhostList",self.EUstatGhostList,flush=True)


    
    # supply 
    def supply(self):

        self.count_months += 1
        
        # here we use 28 but the last positiion of the vector can be an empty space 
        
        self.countWorkers = [0] * 28
        self.countEmployeeCompensations = [0] * 28
        self.countCapitalR = [0] * 28
        self.countActualAddedValue = [0] * 28
        self.countAFF_Firms = [0] * 28
        self.countAFF_WorkerNumber = [0] * 28
        self.intermediateTotalSupplyBySector = [0] * 64
        self.intermediateTotalSupplyByCountry = [0] * 28
        self.intermediateTotalSupplyBySectorAndCountry =  [[0] * 28 for _ in range(64)]
        self.intBySectors=[0]*64
        self.addValBySectors=[0]*64
        self.markupTentative= params['plannedMarkup']


        for aFirm in context.agents(agent_type=0):
      
        ###################################################################################################
        ########################### COMPUTING ADDED VALUE BY FIRM IN EACH RANK ############################
        ###################################################################################################
            
            self.countWorkers[aFirm.country] += aFirm.labor
            self.countEmployeeCompensations[aFirm.country] += (aFirm.labor*aFirm.wage) / 12
            self.countCapitalR[aFirm.country] += (aFirm.capital*aFirm.capitalR) / 12
            if int(aFirm.sector) <= 3: 
                self.countAFF_Firms[aFirm.country] += 1
                self.countAFF_WorkerNumber[aFirm.country] +=aFirm.labor
                
            aFirm.targetAddedValue= float(aFirm.labor * aFirm.wage + aFirm.capital * 1000 * aFirm.capitalR) / 12   # float to avoid np.float64 in vector
            aFirm.targetAddedValue *= self.markupTentative 

            
    
            # firms account for the availability of EACH intermediate good in the intermediate warehouse to produce
            # we have to consider the quantity (in value) that we need for each intermediate good to produce our desired added value
            
            # if this quantity is not available (or partially not available) the firm will limit the production to the quantity 
                                                                                              # of intermediate goods available
            # for simiplicity, we consider for this purpose only intermediate goods evaluated at least at 1% of the firm added value
                
            # notice that firms usually use multiple intermediate goods
            # if many of them are unavailable we have to consider the lower quantity available for the firm production in that cycle 
            # worst case scenario q of one intermediate good = 0 ---> production = 0
        
            
            aFirm.actualAddedValue = aFirm.targetAddedValue
            for j in range(64):
            # what is doing: cutting actualAddedValue
                if aFirm.intermediateWarehouse[j] < (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])* 0.99\
                                                                    and self.intermediateInputs[j][aFirm.sector-1] > 0.001:
                    aFirm.actualAddedValue *= aFirm.intermediateWarehouse[j]/ (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])

            
            # after determining aFirm.actualAddedValue across all 64 sectors we can withdraw the corresponding 
                                                        # intermediate goods quotas from their warehouse

            #if aFirm.targetAddedValue - aFirm.actualAddedValue > 0.001: print(aFirm.uid, aFirm.sector-1)


            # sum of all the actual added values
            self.countActualAddedValue[aFirm.country] += aFirm.actualAddedValue

            intermediateWarehouseWithdraw = 0
            for j in range(64):
                aFirm.intermediateWarehouse[j] -= (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
                intermediateWarehouseWithdraw += (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])


            ############################ intermediate goods firm production ###############################
            if aFirm.productionType == "Int": 
                # totalizers to measure this quantity in a matrix sector x country
                self.intermediateTotalSupplyBySector[aFirm.sector-1] +=  aFirm.actualAddedValue + intermediateWarehouseWithdraw
                self.intermediateTotalSupplyByCountry[aFirm.country] +=  aFirm.actualAddedValue + intermediateWarehouseWithdraw
                self.intermediateTotalSupplyBySectorAndCountry[j][aFirm.country] += aFirm.actualAddedValue + intermediateWarehouseWithdraw


        # should include: a) direct imports from buyers from those sectors, b) the valorization of intermediate products ???
        
        ###################################################################################################
        ###################### SENDING PRODUCTS TO WAREHOUSES BY FIRM IN EACH RANK ########################
        ###################################################################################################
            aFirm.outputWarehouse += aFirm.actualAddedValue + intermediateWarehouseWithdraw 
            # must include a) intermediate goods employed in production; b) imports by producing firm/sector
            
        #print("Supply", self.intermediateTotalSupplyBySector)                
        


    def demand(self): 

        self.firmSubstitutions= [0] * 28
        self.substitutionRate =0.1
        self.substitutionRateL=0.01667 #for sectors 44 and 45, 60 years of duration
        #self.allfirmSubstitutionsByVendorSector=[0]*65
        self.allfirmSubstitutionsByVendorSector=np.zeros([65,28])
        self.allfirmSubstitutionsByBuyerSector=np.zeros([65,28])
        self.intermediateTotalDemandByCountry = [0] * 28
        self.intermediateTotalDemandBySector = [0] * 64
        self.intermediateTotalDemandBySectorAndCountry =  [[0] * 28 for _ in range(64)]
        
        for aFirm in context.agents(agent_type=0):
    
        ###################################################################################################
        ############### CAPITAL SUBSTITUIONS GOODS PURCHASING ORDERS BY FIRM IN EACH RANK #################
        ###################################################################################################
        # here what matters is the substitutions by buyer, whereas those by vendor are relevant only for statistical purposes for check
        
            # imputed rents (special case)
            if aFirm.sector == 44: 
                # from ff_with_class_limits, we use the number of the sector not the position!
                aFirm.capital= (30_000_000 * 1000 * self.eu27_owners.iloc[-1,aFirm.country]) / 12
                
                # the whole EU capital of the sector, to be subdivided by countries
                # see structuralData > firm-features-generation for explanation, cell "Imputed rents special case"
                self.firmSubstitutions[aFirm.country] += ( ((aFirm.capital*self.substitutionRateL)/self.Eu_FirmNumber)\
                                                           *self.simulationFirmsExAnteNumber) / 12
                                        # it will be reported to the EU scale
                    
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += ( ((aFirm.capital*self.substitutionRateL)\
                                                                         /self.Eu_FirmNumber)*self.simulationFirmsExAnteNumber) / 12

                # buying uniquely from constructions (ie sector 27!)
                self.allfirmSubstitutionsByVendorSector[27-1, aFirm.country]+= ( (aFirm.capital*self.substitutionRateL/self.Eu_FirmNumber)*\
                                                                                                  self.simulationFirmsExAnteNumber) / 12

            # real estate (special case 2)
            elif aFirm.sector == 45: # from ff_with_class_limits, we use the number of sector not the position!
                self.firmSubstitutions[aFirm.country] +=  (aFirm.capital*self.substitutionRateL) / 12
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += (aFirm.capital*self.substitutionRateL) / 12
                    
                for s in range(1,66):
                    self.allfirmSubstitutionsByVendorSector[s-1, aFirm.country]+= (aFirm.capital*self.substitutionRateL*\
                                                                                   self.invSharesNp[aFirm.sector-1,s]) / 12
            
            # general case
            else:                                        
                self.firmSubstitutions[aFirm.country] += (aFirm.capital*self.substitutionRate) / 12
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += (aFirm.capital*self.substitutionRate) / 12

                for s in range(1,66):
                    self.allfirmSubstitutionsByVendorSector[s-1, aFirm.country]+= (aFirm.capital*self.substitutionRate*\
                                                                                 self.invSharesNp[aFirm.sector-1,s]) / 12


        ###################################################################################################
        ###################### INTERMEDIATE GOODS PURCHASING NEEDS BY FIRM IN EACH RANK ###################
        ###################################################################################################

        # this step identifies the desired quantities of intermediate inputs in the case that firms can actually produce their entire added value
        # these quantities will be further compared with the availability of the intermediate inputs
        

            for s in range(64):
                aFirm.intermediateWarehouseRefill[s] = aFirm.targetAddedValue * self.intermediateInputs[s][aFirm.sector-1]\
                                                                                         - aFirm.intermediateWarehouse[s]
                # totalizers to measure this quantity in a matrix sector x country
                self.intermediateTotalDemandBySector[s] +=  aFirm.intermediateWarehouseRefill[s] 
                self.intermediateTotalDemandByCountry[aFirm.country] +=  aFirm.intermediateWarehouseRefill[s] 
                self.intermediateTotalDemandBySectorAndCountry[s][aFirm.country] +=  aFirm.intermediateWarehouseRefill[s] 

                
        #print("Demand", self.intermediateTotalDemandBySector)

        # we compared the results of the model computing both the intermediate goods demand and the supply (computed either by sector and by country)
        # follow Intermediate Goods Consistency test at https://prsm.uk/prsm.html?room=UTZ-JPY-SWQ-LJH
        # remember that quantities have been adjusted for the propFactor and from months to year ( x12)
        # the difference between supply and demand of Intermediate goods represents its Export
        

        
        ###################################################################################################
        ############################## COMPUTING GDP BY COUNTRY BY RANK ################################### 
        ###################################################################################################
        # from now we are out of the aFirm "for" cycle because we are observing aggregate consumptions

        self.gdp_eu27 =[0] * 27
        for aCountry in range(27):
            self.gdp_eu27[aCountry] = int(self.markupTentative *\
                                         (self.countEmployeeCompensations[aCountry] * self.propFactor / 1000000\
                                          + self.countCapitalR[aCountry] * self.propFactor / 1000))
            #self.markupTentative * (self.countEmployeeCompensations[aCountry] + self.countCapitalR[aCountry])

        # for jjj in range (27):
            # print(jjj+1,self.eu_countries[jjj],self.gdp_eu27[jjj])


        ###################################################################################################
        ####################### EVALUATING PRIVATE AND GOVERNMENT CONSUMPTIONS ############################ 
        ###################################################################################################
        
        self.private_consumption =[0] * 27
        self.gov_consumption =[0] * 27


        adjusting_consumptions_with_d_imports_private = 1 - 476/7284
        adjusting_consumptions_with_d_imports_gov = 1 - 44/3375
        # these values are copied and rounded from table naio 2022 (data_import.ipynb) to get the parameters providing the adjustment 

        for aCountry in range(27):
            self.private_consumption[aCountry] = self.consumptionShares.iloc[aCountry, -2] * self.gdp_eu27[aCountry]  
            #* adjusting_consumptions_with_d_imports_private ## dubbio: eliminiamo le importazioni dirette per ordinare la prod alle firm?
            self.gov_consumption[aCountry] = self.consumptionShares.iloc[aCountry , - 1] * self.gdp_eu27[aCountry]  
            #* adjusting_consumptions_with_d_imports_gov

        # illuminazione(??): quello che ordiniamo alle imprese produttrici di beni di consumo è la fetta di added vaoue prodotto da loro

        # to calculate internal consumption we introduce two adjusting_consumptions_with_d_imports parameters 
        # neglecting the consumption goods which are directly exported

        # notice that the results produced by the model are very consistent among them
        # let us consider Austria as an example: 
        # simulated gdp = 424188; eurostat gdp (from self.consumptionShares) = 449382.2 --> ratio = 0.94
        # sim private consumption = 16833 * 12 =  201996 ; eurostat private consumption (from self.consumptionShares) = 228963.2 -->  ratio = 0.94
        # meaning that they are perfectly in fit reminding that direct exports are a share included in the general private consumption

        # lastly, notice that direct exports may represent significant tools for policy decisions



        
    ### -------------------------------- SYNCH TO GHOSTS (will require to define a new fct)
                

    ###################### INTERMEDIATE GOODS RANDOM SUPPLIER SELECTION ###########################
    def selectSuppliers(self):

        #CHE COSA VA UP UNA TANTUM row 371
        
        # int firm only
        self.probabilitiesOfChoiceBySectorAndCountry = [[ [] for _ in range(28)] for _ in range(64)]
        self.firmListBySectorAndCountryWithProductionTypeInt = [[ [] for _ in range(28)] for _ in range(64)]
        
        for sector in range(64):
            for aFirm in self.firmListBySectorWithProductionTypeInt[sector]:
                for country in range(28):
                    coeff = 1
                    if aFirm.country == country: coeff = params['localPreference']
                    self.probabilitiesOfChoiceBySectorAndCountry[sector][country].append(aFirm.actualAddedValue * coeff)
                    self.firmListBySectorAndCountryWithProductionTypeInt[sector][country].append(aFirm)

        for sector in range(64):
            for country in range(28):
                self.cumProbabilitiesOfChoiceBySectorAndCountry =\
                                        list(accumulate(self.probabilitiesOfChoiceBySectorAndCountry))
        
        conteggio=0
        for aFirm in context.agents(agent_type=0):    
            for sector in range(64):
                if sector != 43: continue
                supplier = random.choices(self.firmListBySectorAndCountryWithProductionTypeInt[sector][aFirm.country],\
                            cum_weights = self.cumProbabilitiesOfChoiceBySectorAndCountry[sector][aFirm.country],k=1)[0]
                if aFirm.sector == 3: print(aFirm.uid, aFirm.country, aFirm.sector, supplier.uid, supplier.country, supplier.sector,\
                                                                                    supplier.outputWarehouse)
            conteggio+=1
            if conteggio % 1000==0: print(conteggio)
        1/0
            
        # first, we introduce the dices which provide the randomicity of the supplier selection process
        # then, we will fill the warehouse with the purchases performed with the dices

    
    def aggregatingResultsByCountry(self):
        for j in range(27):
            self.countWorkers[27] += self.countWorkers[j]
            self.countEmployeeCompensations[27] += self.countEmployeeCompensations[j] 
            self.countCapitalR[27] += self.countCapitalR[j] 
            self.countActualAddedValue[27] += self.countActualAddedValue[j]
            self.countAFF_Firms[27] += self.countAFF_Firms[j]
            self.countAFF_WorkerNumber[27] += self.countAFF_WorkerNumber[j] 
            self.intermediateTotalSupplyByCountry[27] += self.intermediateTotalSupplyByCountry[j]
            self.firmSubstitutions[27] += self.firmSubstitutions[j] 

            self.allfirmSubstitutionsByVendorSector[:, 27] += self.allfirmSubstitutionsByVendorSector[:,j]
            self.allfirmSubstitutionsByBuyerSector[:, 27] += self.allfirmSubstitutionsByBuyerSector[:,j]

            # remember that you have to add the aggregation by sector!!!!!!!!!!!!!!!!!!!!!!


    # fill the list self.rankResults
    def fillingRankResults(self):
        if rank == 0:
            return
            
        rankEUstat_uid = (0,1,rank)
        rankEUstat = context.agent(rankEUstat_uid)  
        rankEUstat.rankResults = []
        rankEUstat.rankResults.append(self.countActualExPostFirmNumber) # rankEUstat.rankResults[0]

        rankEUstat.rankResults.append(self.countWorkers) # rankEUstat.rankResults[1]
        rankEUstat.rankResults.append(self.countEmployeeCompensations) # rankEUstat.rankResults[2]
        rankEUstat.rankResults.append(self.countCapitalR) # rankEUstat.rankResults[3]
        rankEUstat.rankResults.append(self.countAFF_Firms) # rankEUstat.rankResults[4]
        rankEUstat.rankResults.append(self.countAFF_WorkerNumber) # rankEUstat.rankResults[5]
        rankEUstat.rankResults.append(self.intermediateTotalSupplyBySector) # rankEUstat.rankResults[6]
        rankEUstat.rankResults.append(self.intermediateTotalSupplyByCountry) # rankEUstat.rankResults[7]
        rankEUstat.rankResults.append(self.intermediateTotalDemandBySector) # rankEUstat.rankResults[8]
        rankEUstat.rankResults.append(self.intermediateTotalDemandByCountry) # rankEUstat.rankResults[9]

        rankEUstat.rankResults.append(self.firmSubstitutions) # rankEUstat.rankResults[10]
        rankEUstat.rankResults.append(self.allfirmSubstitutionsByVendorSector) # rankEUstat.rankResults[11] - matrix
        rankEUstat.rankResults.append(self.allfirmSubstitutionsByBuyerSector) # rankEUstat.rankResults[12] - matrix
        rankEUstat.rankResults.append(self.intermediateTotalSupplyBySectorAndCountry) # rankEUstat.rankResults[13] - matrix
        rankEUstat.rankResults.append(self.intermediateTotalDemandBySectorAndCountry) # rankEUstat.rankResults[14] - matrix

        rankEUstat.rankResults.append(self.countActualAddedValue) # rankEUstat.rankResults[15] - vector
        #rankEUstat.rankResults.append(self.) # rankEUstat.rankResults[]

        #print(rank, rankEUstat.rankResults)

    def synchronizeRanks(self):
        if rankNum > 1: context.synchronize(restore_agent)
    
    def aggregatingRanks(self):
        if rankNum == 1: return
        if rank != 0: return         
            
        if self.count_months == 0: self.countActualExPostFirmNumber += scalarSummingResultsOverRanks(0)
        L = 28
        for j in range(28):
            self.countWorkers[j] += summingResultsOverRanks(1, L)[j]   
            self.countEmployeeCompensations[j] += summingResultsOverRanks(2, L)[j]
            self.countCapitalR[j] += summingResultsOverRanks(3, L)[j]
            self.countAFF_Firms[j] += summingResultsOverRanks(4, L)[j]
            self.countAFF_WorkerNumber[j] += summingResultsOverRanks(5, L)[j]
            self.intermediateTotalSupplyByCountry[j] += summingResultsOverRanks(7, L)[j]
            self.intermediateTotalDemandByCountry[j] += summingResultsOverRanks(9, L)[j]
            self.firmSubstitutions[j] += summingResultsOverRanks(10, L)[j]
            self.countActualAddedValue[j] += summingResultsOverRanks(15, L)[j]

        LL = 64
        for k in range(len(self.intermediateTotalSupplyBySector)):
            self.intermediateTotalSupplyBySector[k] += summingResultsOverRanks(6, LL)[k]
            self.intermediateTotalDemandBySector[k] += summingResultsOverRanks(8, LL)[k]

        for j in range(28):
            for k in range(len(self.allfirmSubstitutionsByVendorSector)):
                self.allfirmSubstitutionsByVendorSector[k,j] += summingNumpyMatrixResultsOverRanks(11)[k,j]
                self.allfirmSubstitutionsByBuyerSector[k,j] += summingNumpyMatrixResultsOverRanks(12)[k,j]
                
            for k in range(len(self.intermediateTotalSupplyBySectorAndCountry)):
                self.intermediateTotalSupplyBySectorAndCountry[k][j] += summingMatrixResultsOverRanks(13)[k][j]
                self.intermediateTotalDemandBySectorAndCountry[k][j] += summingMatrixResultsOverRanks(14)[k][j]
        
        
    # show results
    def showResults(self):
        
        if rank != 0: return

        self.propFactor = self.Eu_FirmNumber / self.countActualExPostFirmNumber # redefined here to be updated with rankNum > 1

        print("\n", "MONTH",  (self.count_months % 12) + 1 , "YEAR", (self.count_months // 12) + 1)
        
        generateOutput("totEU27", sys.stdout, self.AFF_FirmNumber, self.countAFF_Firms[27], self.propFactor, self.AFF_WorkerNumber,\
                    self.countAFF_WorkerNumber[27], self.Eu_FirmNumber, self.countActualExPostFirmNumber, self.Eu_WorkerNumber,\
                    self.countWorkers[27], self.Eu_EmployeeCompensations, self.countEmployeeCompensations[27], self.Eu_GDP_withVAT2022,\
                    self.Eu_AddedValue2022, self.countCapitalR[27], self.countActualAddedValue[27], self.markupTentative, self.Eu_Intermediate2022,\
                       self.intermediateTotalSupplyByCountry[27], self.substitutionRate, self.substitutionRateL, self.firmSubstitutions[27],\
                       self.allfirmSubstitutionsByBuyerSector[:,27], self.simulationFirmsExAnteNumber, self.allfirmSubstitutionsByVendorSector[:,27])

        if self.selected_countries == []: return

        for aCountry in self.selected_countries:
            if os.path.exists(self.eu_countries[aCountry]+".txt") and self.count_months == 1: os.remove(self.eu_countries[aCountry]+".txt")
            
            with open(self.eu_countries[aCountry]+'.txt', 'a', newline='') as txtfile:              
                #print(txtfile)
                print("MONTH", (self.count_months % 12) + 1, file = txtfile)
                generateOutput(self.eu_countries[aCountry], txtfile, self.AFF_FirmNumber,
                                self.countAFF_Firms[aCountry], self.propFactor,
                                self.AFF_WorkerNumber, self.countAFF_WorkerNumber[aCountry],
                                self.Eu_FirmNumber, self.countActualExPostFirmNumber,
                                self.Eu_WorkerNumber, self.countWorkers[aCountry],
                                self.Eu_EmployeeCompensations, self.countEmployeeCompensations[aCountry],
                                self.Eu_GDP_withVAT2022, self.Eu_AddedValue2022,
                                self.countCapitalR[aCountry], 
                                self.countActualAddedValue[aCountry],
                                self.markupTentative,
                                self.Eu_Intermediate2022, self.intermediateTotalSupplyByCountry[aCountry],
                                self.substitutionRate, self.substitutionRateL,
                                self.firmSubstitutions[aCountry],
                                self.allfirmSubstitutionsByBuyerSector[:, aCountry],
                                self.simulationFirmsExAnteNumber,
                                self.allfirmSubstitutionsByVendorSector[:, aCountry]
                              )

        # ptpt if rank == 0: 
        print("\n", 'Add. Val. basic p. by country', "\n")
        for aCountry in range(len(self.eu_countries)):
            generateGDPOutput(self.eu_countries[aCountry], sys.stdout, self.propFactor, self.countActualAddedValue[aCountry])
                        #self.countEmployeeCompensations[aCountry], self.countCapitalR[aCountry], self.markupTentative)

    
    #finish
    def finish(self):
        print("\n\n")
        print(f"Rank {rank}, gloabal time {T()}, cpu time {Tc()}")
        print("\nRank ", rank, "concluded")
    
    def start(self):
        runner.execute()

def run():

    model = Model() 
    model.start()
    
run()

#file name: ff_with_class_limits.csv

rank 0 tableOfInvestmentShare 2
rank 0 how many firms 1000000
rank 0 use smart capital 1
1000031 firms created in rank  0
propFactor 42.25056423250879 in rank 0
Ci sono 64 liste.
La lista 1 contiene 51452 elementi.
La lista 2 contiene 5163 elementi.
La lista 3 contiene 4914 elementi.
La lista 4 contiene 21 elementi.
La lista 5 contiene 4404 elementi.
La lista 6 contiene 1159 elementi.
La lista 7 contiene 99 elementi.
La lista 8 contiene 244 elementi.
La lista 9 contiene 65 elementi.
La lista 10 contiene 289 elementi.
La lista 11 contiene 431 elementi.
La lista 12 contiene 746 elementi.
La lista 13 contiene 306 elementi.
La lista 14 contiene 178 elementi.
La lista 15 contiene 3 elementi.
La lista 16 contiene 276 elementi.
La lista 17 contiene 611 elementi.
La lista 18 contiene 578 elementi.
La lista 19 contiene 145 elementi.
La lista 20 contiene 2050 elementi.
La lista 21 contiene 210 elementi.
La lista 22 contiene 1378 elementi.
La lista 23 contiene 27 elementi.
La lista 24 contiene 78

ZeroDivisionError: division by zero

Italy simulated GDP per month: 133108
Italy GDP in 2022: 1946479 --> 1946479 / (12 * 1.22) = 132956.2 ca. to get monthly at basic price GDP 

**to run the code in a multi-rank way, from the terminal launch:**  

mpirun -n x ipython modelEU27.ipynb